In [2]:
import pandas as pd
import glob

path = f'data/*/*.csv'

dataframes = []
for p in glob.glob(path):
    df = pd.read_csv(p)
    dataframes.append(df)

df = pd.concat(dataframes, ignore_index=True)
df.head()

,id,author,created_utc,date_utc,subreddit,title,selftext,arabic_ratio
0,1u9px60,Practical_Finger7349,1781836817,2026-06-19,jordan,سامحوني عالنكد بس مدري مالي,^(والله يخوان اصعب صراع الي يكون بينك وبينك),0.950
1,1u9piso,Practical_Finger7349,1781835645,2026-06-19,jordan,عميق,تعرف شعور انك ندمان ليش ما بلشت بشغله زمان ترا...,0.987
2,1u9pd23,Practical_Finger7349,1781835190,2026-06-19,jordan,سؤال صعب,سؤال يستحق انك توقف وتصفن بيه \n\nهل انت عايش ...,1.000
3,1u9p2s2,Practical_Finger7349,1781834372,2026-06-19,jordan,خيال ولا واقع؟,يخوان انا الحمد لله مثقف وافهم يعني مش من النو...,0.994
4,1u9oowo,TopReport133,1781833225,2026-06-19,jordan,Jordanian channel for kids bil 3amiye,ولادنا بالغربة عم يكبروا والعربي عم يصير غريب ...,0.733


In [3]:
claude_scrapper = df
carma_data = pd.read_csv('~/data/cleaned_reddit_data.csv')
carma_data.head()

,selftext,created_utc,id,author,subreddit,is_comment
0,يا ساتر واضحه اثار الكبت علينا,1704964515,khc9428,Turkipe,r_SaudiForSaudis,True
1,الطايف هي تركيا الثانيه اذا تبي تنضرب انت وهي ...,1704964612,khc98j4,Double-Stuff6831,r_SaudiForSaudis,True
2,والله ياخي على كثر ما رحت للطايف وكل مره تكون ...,1704964711,khc9d2q,Turkipe,r_SaudiForSaudis,True
3,حياك معاك سعودية ودي اقولك ترا الاشاعات صدق، ك...,1704964785,khc9gjh,Turkipe,r_SaudiForSaudis,True
4,هذي بدايتها؟ ما امدانا نقول ما نبي احد ينزل عن...,1704964995,khc9qmt,Double-Stuff6831,r_SaudiForSaudis,True


In [4]:
len(carma_data)

18701853

In [5]:
unique_claude_posts = claude_scrapper[
    ~claude_scrapper['id'].isin(carma_data['id'])
]
unique_claude_posts.head()

,id,author,created_utc,date_utc,subreddit,title,selftext,arabic_ratio
0,1u9px60,Practical_Finger7349,1781836817,2026-06-19,jordan,سامحوني عالنكد بس مدري مالي,^(والله يخوان اصعب صراع الي يكون بينك وبينك),0.950
1,1u9piso,Practical_Finger7349,1781835645,2026-06-19,jordan,عميق,تعرف شعور انك ندمان ليش ما بلشت بشغله زمان ترا...,0.987
2,1u9pd23,Practical_Finger7349,1781835190,2026-06-19,jordan,سؤال صعب,سؤال يستحق انك توقف وتصفن بيه \n\nهل انت عايش ...,1.000
3,1u9p2s2,Practical_Finger7349,1781834372,2026-06-19,jordan,خيال ولا واقع؟,يخوان انا الحمد لله مثقف وافهم يعني مش من النو...,0.994
4,1u9oowo,TopReport133,1781833225,2026-06-19,jordan,Jordanian channel for kids bil 3amiye,ولادنا بالغربة عم يكبروا والعربي عم يصير غريب ...,0.733


In [6]:
all_reddit = pd.concat([unique_claude_posts, carma_data], ignore_index=True)
all_reddit.head()

,id,author,created_utc,date_utc,subreddit,title,selftext,arabic_ratio,is_comment
0,1u9px60,Practical_Finger7349,1781836817,2026-06-19,jordan,سامحوني عالنكد بس مدري مالي,^(والله يخوان اصعب صراع الي يكون بينك وبينك),0.950,NaN
1,1u9piso,Practical_Finger7349,1781835645,2026-06-19,jordan,عميق,تعرف شعور انك ندمان ليش ما بلشت بشغله زمان ترا...,0.987,NaN
2,1u9pd23,Practical_Finger7349,1781835190,2026-06-19,jordan,سؤال صعب,سؤال يستحق انك توقف وتصفن بيه \n\nهل انت عايش ...,1.000,NaN
3,1u9p2s2,Practical_Finger7349,1781834372,2026-06-19,jordan,خيال ولا واقع؟,يخوان انا الحمد لله مثقف وافهم يعني مش من النو...,0.994,NaN
4,1u9oowo,TopReport133,1781833225,2026-06-19,jordan,Jordanian channel for kids bil 3amiye,ولادنا بالغربة عم يكبروا والعربي عم يصير غريب ...,0.733,NaN


In [7]:
all_reddit["text"] = (
    all_reddit["selftext"].fillna("") + " " +
    all_reddit["title"].fillna("")
).str.strip()

In [8]:
all_reddit['text'].describe()

count                                              18939794
unique                                             18189466
top       Welcome to r/Morocco! Please always make sure ...
freq                                                  66066
Name: text, dtype: object

In [9]:
# clean english
import tnkeeh as tn

# cleaner = tn.Tnkeeh(remove_english=True,
#                     remove_diacritics=True,
#                     remove_html_elements=True,
#                     remove_links=True,
#                     remove_special_chars=True,
#                     remove_tatweel=True)
# cleaned_dataset = cleaner.clean_data_frame(all_reddit, 'text')
import RedditCleaner

cleaner = RedditCleaner.RedditCleaner(
    min_arabic_ratio=0.50,  # stricter — fewer mixed posts survive
    min_chars=50,           # longer minimum post length
    strip_english=True,     # set False to keep English words (not recommended)
)
cleaned_dataset = cleaner.clean_data_frame(all_reddit, 'text')

cleaned_dataset.to_csv('~/data/reddit-corpus-reddit-cleaner-cleaned.csv')



════════════════════════════════════════════════════
  RedditCleaner — Corpus Stats
════════════════════════════════════════════════════
  Raw rows              : 18,939,794
  Removed (empty/null)  :          0
  Removed ([deleted])   :          0
  Removed (spaced AR)   :        760
  Removed (punct only)  : 11,916,915
  Removed (low Arabic)  :          0
  Removed (too short)   :  2,351,651
  Removed (numbers only):          0
  ─────────────────────────────────────
  Total removed         : 14,269,326  (75.3%)
  Final corpus size     :  4,670,468  (24.7%)
  ─────────────────────────────────────
  Chars before          : 4,027,096,974
  Chars after           : 1,021,920,741
  Char retention        :      25.4%
  English words removed : 501,067,877
  URLs removed          :     13,341
  Quranic chars stripped:    241,940
════════════════════════════════════════════════════



In [10]:
cleaned_dataset.sample(n=500)['text'].to_csv('sample-reddit.csv')

In [11]:
len(cleaned_dataset)

4670468